In [0]:
import json
import os
import re

dbutils.widgets.dropdown("mode", "dry_run", ["dry_run", "apply"])
dbutils.widgets.text(
    "catalog_path",
    "/Workspace/Shared/RMIT_CDR_Capstone/04_Governance/rbac/role_catalog.json",
)
dbutils.widgets.dropdown("force", "false", ["false", "true"])

MODE = dbutils.widgets.get("mode")
CATALOG_PATH = dbutils.widgets.get("catalog_path")
FORCE = dbutils.widgets.get("force") == "true"
assert MODE in ("dry_run", "apply"), MODE


def load_catalog(path):
    """Load the role catalog from the workspace path, or from the repo checkout."""
    candidates = [
        path,
        "04_Governance/rbac/role_catalog.json",
        os.path.join(os.getcwd(), "04_Governance/rbac/role_catalog.json"),
    ]
    tried = []
    for candidate in candidates:
        if not candidate:
            continue
        tried.append(candidate)
        if os.path.exists(candidate):
            with open(candidate, "r", encoding="utf-8") as handle:
                return json.load(handle), candidate
    raise FileNotFoundError("role_catalog.json not found. Looked in: " + ", ".join(tried))


CATALOG_DEF, CATALOG_SOURCE = load_catalog(CATALOG_PATH)
CATALOG = CATALOG_DEF["catalog"]
GOLD_SCHEMA = CATALOG_DEF["gold_schema"]
ROLES = CATALOG_DEF["roles"]
print(f"Loaded {len(ROLES)} roles from {CATALOG_SOURCE}")

IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")

for role_name, spec in ROLES.items():
    assert IDENT.match(role_name), f"invalid group name: {role_name}"
    for schema_name in spec["schemas"]:
        assert schema_name in ("bronze", "silver", "gold"), schema_name

Loaded 7 roles from /Workspace/Shared/RMIT_CDR_Capstone/04_Governance/rbac/role_catalog.json


## 1. Safety check

Provisioning is an admin operation. Stop unless the caller is in `cdr_admins`
(or explicitly forces it).

In [0]:
def is_member(group):
    """Unity Catalog membership test.

    `is_member` resolves account groups and workspace-local groups; the older
    `is_account_group_member` only resolves account groups, which is why the two
    notebooks used to disagree about who was an admin.
    """
    try:
        return bool(spark.sql(f"SELECT is_member('{group}')").first()[0])
    except Exception:
        return bool(spark.sql(f"SELECT is_account_group_member('{group}')").first()[0])


IS_ADMIN = is_member("cdr_admins")
assert IS_ADMIN or FORCE, (
    "Run this as a member of cdr_admins, or set the force widget to true "
    "(you need MANAGE/ownership on the catalog either way)."
)
print(f"admin={IS_ADMIN} force={FORCE} mode={MODE}")

admin=True force=True mode=apply


## 2. Build the statements

One expected statement per privilege in the catalog. Domain-scoped roles
(`cdr_pharmacy_analyst`, `cdr_external_researcher`) get table-level `SELECT`
resolved from the `domain` tag that `01_apply_tags` applies — so the tag
vocabulary and the catalog domains must agree.

In [0]:
# domain -> set of gold tables, from the governed tag taxonomy
domain_rows = spark.sql(f"""
    SELECT g.table_name, g.tag_value
    FROM {CATALOG}.information_schema.table_tags g
    WHERE g.schema_name = '{GOLD_SCHEMA}' AND g.tag_name = 'domain'
""").collect()

tables_by_domain = {}
for row in domain_rows:
    tables_by_domain.setdefault(str(row["tag_value"]), set()).add(str(row["table_name"]))

print(f"{len(domain_rows)} domain tags across {len(tables_by_domain)} domains: "
      + ", ".join(f"{d}={len(t)}" for d, t in sorted(tables_by_domain.items())))

for role_name, spec in ROLES.items():
    if "ALL" in spec["domains"]:
        continue
    for domain in spec["domains"]:
        assert IDENT.match(domain), f"invalid domain '{domain}' for {role_name}"
        if not tables_by_domain.get(domain):
            print(f"WARNING: {role_name} needs domain '{domain}' but no gold table carries that tag")

28 domain tags across 7 domains: clinical_coding=4, critical_care=4, demographics=1, diagnostics=4, encounter=8, orders=3, pharmacy=4


In [0]:
def q(name):
    return "`" + name + "`"


def build_statements(catalog_def):
    """Return [(label, sql)] for every grant in the catalog. Pure function, no SQL run."""
    catalog = catalog_def["catalog"]
    statements = []

    for role_name, spec in catalog_def["roles"].items():
        role = q(role_name)
        whole_layer = "ALL" in spec["domains"]
        statements.append(
            (f"{role_name}: use catalog",
             f"GRANT USE CATALOG ON CATALOG {catalog} TO {role}")
        )

        for schema_name, privileges in spec["schemas"].items():
            statements.append(
                (f"{role_name}: use {schema_name}",
                 f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema_name} TO {role}")
            )
            if not whole_layer:
                # domain-scoped roles get table grants only; a schema-level SELECT here
                # would hand them the whole layer and void the least-privilege design
                continue
            for privilege in privileges:
                assert privilege in ("SELECT", "MODIFY"), privilege
                statements.append(
                    (f"{role_name}: {privilege.lower()} {schema_name}",
                     f"GRANT {privilege} ON SCHEMA {catalog}.{schema_name} TO {role}")
                )

        if "ALL" not in spec["domains"]:
            for domain in spec["domains"]:
                for table_name in sorted(tables_by_domain.get(domain, set())):
                    statements.append(
                        (f"{role_name}: select {table_name} ({domain})",
                         f"GRANT SELECT ON TABLE {catalog}.{GOLD_SCHEMA}.{q(table_name)} TO {role}")
                    )

    return statements


STATEMENTS = build_statements(CATALOG_DEF)
print(f"{len(STATEMENTS)} grant statements prepared for {len(ROLES)} roles")

41 grant statements prepared for 7 roles


## 3. Create missing groups and apply

`SHOW GROUPS` lists workspace-local groups only, so it is a hint rather than
the truth; the real test is whether the grant succeeds.

In [0]:
try:
    existing_groups = {str(row.name) for row in spark.sql("SHOW GROUPS").collect()}
except Exception as exc:  # listing groups needs admin on some workspaces
    existing_groups = set()
    print("NOTE: could not list groups:", str(exc).splitlines()[0][:160])

missing_groups = [r for r in ROLES if r not in existing_groups]
print(f"{len(missing_groups)} of {len(ROLES)} groups not visible as workspace-local groups")

created, already, errors = [], [], []

for role_name in missing_groups:
    sql = f"CREATE GROUP {q(role_name)}"
    if MODE == "dry_run":
        print(sql)
        continue
    try:
        spark.sql(sql)
        created.append(role_name)
    except Exception as exc:
        message = str(exc)
        if "exist" in message.lower():
            already.append(role_name)
        else:
            errors.append((role_name, message.splitlines()[0][:200]))

print(f"groups created={len(created)} already-present={len(already)} errors={len(errors)}")

0 of 7 groups not visible as workspace-local groups
groups created=0 already-present=0 errors=0


In [0]:
applied = 0

for label, sql in STATEMENTS:
    if MODE == "dry_run":
        print(sql)
        continue
    try:
        spark.sql(sql)
        applied += 1
    except Exception as exc:
        errors.append((label, str(exc).splitlines()[0][:200]))

print(f"\nmode={MODE} statements={len(STATEMENTS)} applied={applied} errors={len(errors)}")
for label, message in errors:
    print("ERROR:", label, "->", message)


mode=apply statements=41 applied=41 errors=0


## 4. Summary

In [0]:
summary = [
    {
        "group": role_name,
        "schemas": ", ".join(spec["schemas"].keys()),
        "domains": ", ".join(spec["domains"]),
        "direct_identifier": spec["policy"]["direct_identifier"],
        "hl7_vr": spec["policy"]["hl7_vr"],
        "rls_level": spec["rls"]["level"],
    }
    for role_name, spec in ROLES.items()
]
display(spark.createDataFrame(summary))

if MODE == "apply":
    display(spark.sql(f"""
        SHOW GRANTS ON CATALOG {CATALOG}
    """))

assert not errors, f"{len(errors)} statement(s) failed; see output"
dbutils.notebook.exit(
    f"03_create_groups mode={MODE} roles={len(ROLES)} statements={len(STATEMENTS)} errors={len(errors)}"
)

In [0]:
print(STATEMENTS)

[('cdr_admins: use catalog', 'GRANT USE CATALOG ON CATALOG cdr_mimiciv TO `cdr_admins`'), ('cdr_admins: use bronze', 'GRANT USE SCHEMA ON SCHEMA cdr_mimiciv.bronze TO `cdr_admins`'), ('cdr_admins: select bronze', 'GRANT SELECT ON SCHEMA cdr_mimiciv.bronze TO `cdr_admins`'), ('cdr_admins: modify bronze', 'GRANT MODIFY ON SCHEMA cdr_mimiciv.bronze TO `cdr_admins`'), ('cdr_admins: use silver', 'GRANT USE SCHEMA ON SCHEMA cdr_mimiciv.silver TO `cdr_admins`'), ('cdr_admins: select silver', 'GRANT SELECT ON SCHEMA cdr_mimiciv.silver TO `cdr_admins`'), ('cdr_admins: modify silver', 'GRANT MODIFY ON SCHEMA cdr_mimiciv.silver TO `cdr_admins`'), ('cdr_admins: use gold', 'GRANT USE SCHEMA ON SCHEMA cdr_mimiciv.gold TO `cdr_admins`'), ('cdr_admins: select gold', 'GRANT SELECT ON SCHEMA cdr_mimiciv.gold TO `cdr_admins`'), ('cdr_admins: modify gold', 'GRANT MODIFY ON SCHEMA cdr_mimiciv.gold TO `cdr_admins`'), ('cdr_standard_researcher: use catalog', 'GRANT USE CATALOG ON CATALOG cdr_mimiciv TO `cdr_